# 🧠 Delentia SLM — JITNA v0.3 QLoRA Fine-tuning

**Model**: Llama 3.1 8B → QLoRA fine-tune → GGUF Q4_K_M export  
**Target metrics**: JITNA ≥ 98% · TOON ≥ 95% · FDIA ≥ 0.895 · Hallucination ≤ 0.28% · Token Savings ≥ 15%  
**GPU**: T4 (free) → ~4–6h | A100 (Colab Pro) → ~1.5h

---
### Steps
1. Mount Drive + clone repo
2. Install dependencies
3. Compile v0.3 mixed logic dataset
4. Validate dataset (gate: ≥500 pairs, avg FDIA ≥ 0.7)
5. Fine-tune (QLoRA)
6. Evaluate (gate: all metrics pass)
7. Export GGUF Q4_K_M
8. Upload to HuggingFace Hub
9. Smoke-test with Ollama

In [ ]:
# ─── Cell 1: Mount Google Drive + clone repo ─────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Google Drive mounted successfully.')
except Exception as e:
    print(f'⚠️ Google Drive mount skipped or failed: {e}')
    print('Proceeding with local runtime storage (Google Drive is not required for training).')

import os, subprocess, sys

# Map Colab secrets (including KAGGLE_k fallback) to environment variables
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY") or userdata.get("KAGGLE_k") or ""
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME") or "delentialabs"
    print("✅ environment credentials mapped successfully.")
except Exception as e:
    print(f"Credential mapping warning: {e}")

REPO_URL = 'https://github.com/delentia-labs/Delentia-AI-SLM.git'
REPO_DIR = '/content/Delentia-AI-SLM'
OS_URL = 'https://github.com/delentia-labs/Delentia-OS.git'
OS_DIR = '/content/Delentia-OS'

# Clone SLM Training repo
if not os.path.exists(REPO_DIR):
    result = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True)
    print(result.stdout or result.stderr)
else:
    result = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
    print('SLM repo already exists — pulled latest:', result.stdout.strip())

# Clone OS repo (critical for JITNA pair extraction)
if not os.path.exists(OS_DIR):
    result = subprocess.run(['git', 'clone', OS_URL, OS_DIR], capture_output=True, text=True)
    print(result.stdout or result.stderr)
else:
    result = subprocess.run(['git', '-C', OS_DIR, 'pull'], capture_output=True, text=True)
    print('OS repo already exists — pulled latest:', result.stdout.strip())

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ─── Cell 2: Install dependencies ────────────────────────────────────────────
# Unsloth optimized for T4/A100 — 2x faster training, 60% less VRAM
import subprocess, sys

# Detect GPU type via PyTorch for safety
import torch
if not torch.cuda.is_available():
    print('⚠️ WARNING: GPU runtime is not active! please change Colab runtime to GPU (Runtime -> Change runtime type)')
else:
    gpu_name = torch.cuda.get_device_name(0)
    print(f'✅ GPU detected: {gpu_name}')

# Install Unsloth + training deps
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git',
    '--quiet'
])

# Install local Delentia OS SDK package
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-e', '/content/Delentia-OS', '--quiet'
])

# Install project requirements
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '--quiet'
])

print('✅ All dependencies installed')

In [ ]:
# ─── Cell 3: Compile v0.3 mixed logic dataset ────────────────────────────────
# Invokes the v0.3 dataset generator to synthesize Delta, Loop, and RCT 7 scenarios
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'datasets/scripts/generate_v03_dataset.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('Dataset generation failed')

# Show sample
import json
with open('datasets/processed/jitna_pairs_v03.jsonl') as f:
    lines = f.readlines()
print(f'\n📊 Total pairs in v0.3 dataset: {len(lines)}')
print('Sample pair:')
print(json.dumps(json.loads(lines[0]), indent=2, ensure_ascii=False))

In [ ]:
# ─── Cell 4: Validate dataset (GATE) ─────────────────────────────────────────
# Gate: ≥500 pairs AND average FDIA ≥ 0.70
# Training will NOT proceed if either condition fails.
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'datasets/scripts/validate_dataset.py',
     'datasets/processed/jitna_pairs_v03.jsonl',
     '--toon'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('VALIDATION FAILED — STOPPING')
    print(result.stderr)
    raise SystemExit('Dataset validation gate failed. Add more data before training.')

print('✅ Dataset validation PASSED — proceeding to training')

In [ ]:
# ─── Cell 5: QLoRA Fine-tuning ────────────────────────────────────────────────
# Llama 3.1 8B → QLoRA r=32, alpha=64, target_modules=all-linear
# T4: ~4–6h | A100: ~1.5h
# Checkpoints saved to: models/checkpoints/v0.3_cognitive_kernel/
import subprocess, sys

# Stream stdout/stderr live to Colab cell output
process = subprocess.Popen(
    [sys.executable, 'training/finetune.py',
     '--config', 'training/config/slm_jitna_v0.3.yaml',
     '--toon',
     '--adapter-path', 'models/adapters/jitna_v0.3_toon'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end="")

process.wait()
if process.returncode != 0:
    raise RuntimeError(f"Fine-tuning failed with exit code {process.returncode}")

print('\n✅ Fine-tuning complete — model saved to models/checkpoints/v0.3_cognitive_kernel/')

In [ ]:
# ─── Optional: Backup Adapter to Google Drive ────────────────────────────────
# Saves the completed LoRA adapter to your Google Drive to prevent having to retrain
# if the Colab session disconnects.
import os
if os.path.exists('models/adapters/jitna_v0.3_toon'):
    !cp -r models/adapters/jitna_v0.3_toon /content/drive/MyDrive/jitna_v0.3_toon_backup
    print('✅ Adapter backed up to Google Drive successfully!')
else:
    print('⚠️ Local adapter folder not found. Make sure Cell 5 finished successfully.')

In [ ]:
# ─── Optional: Restore Adapter from Google Drive ──────────────────────────────
# Restores the adapter from your Google Drive so you can skip Cell 5 (Fine-tuning)
# and proceed straight to evaluation, export, or smoke tests.
import os
if os.path.exists('/content/drive/MyDrive/jitna_v0.3_toon_backup'):
    !mkdir -p models/adapters/
    !cp -r /content/drive/MyDrive/jitna_v0.3_toon_backup models/adapters/jitna_v0.3_toon
    print('✅ Adapter restored from Google Drive successfully! You can skip Cell 5 and proceed.')
else:
    print('⚠️ Backup not found on Google Drive. Make sure backup was created at /content/drive/MyDrive/jitna_v0.3_toon_backup.')

In [ ]:
# ─── Cell 6: Evaluate (GATE) ─────────────────────────────────────────────────
# Gate: JITNA ≥ 98% · TOON ≥ 95% · FDIA ≥ 0.895 · Hallucination ≤ 0.28% · Token Savings ≥ 15%
# Export will NOT proceed if any gate fails.
import subprocess, sys

# Stream stdout/stderr live to Colab cell output
process = subprocess.Popen(
    [sys.executable, 'training/evaluate.py',
     '--config', 'training/config/slm_jitna_v0.3.yaml',
     '--eval-data', 'datasets/processed/jitna_pairs_v03.jsonl',
     '--adapter-path', 'models/adapters/jitna_v0.3_toon',
     '--toon'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end="")

process.wait()
if process.returncode != 0:
    print('EVALUATION GATE FAILED — NOT exporting')
    raise SystemExit('Model did not meet minimum quality gates. Retrain with more data or longer epochs.')

print('✅ All quality gates PASSED — proceeding to GGUF export')

In [ ]:
# ─── Cell 7: Export to GGUF Q4_K_M ───────────────────────────────────────────
# Quantizes the fine-tuned model → GGUF format for Ollama + llama.cpp
import subprocess, sys

# Stream stdout/stderr live to Colab cell output
process = subprocess.Popen(
    [sys.executable, 'training/export_gguf.py',
     '--toon',
     '--adapter-path', 'models/adapters/jitna_v0.3_toon',
     '--gguf-path', 'models/gguf/delentia-jitna-v0.3-Q4_K_M.gguf',
     '--model-name', 'delentia-jitna-v0.3'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end="")

process.wait()
if process.returncode != 0:
    raise RuntimeError('GGUF export failed')

import os, glob
gguf_files = glob.glob('models/gguf/*.gguf')
print(f'\n✅ GGUF files created:')
for f in gguf_files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'  {f} ({size_mb:.0f} MB)')

In [ ]:
# ─── Cell 8: Upload to HuggingFace Hub ───────────────────────────────────────
# Requires: HF_TOKEN secret in Colab Secrets (key icon in left sidebar)
import os, glob
from huggingface_hub import login, HfApi

# Read token from Colab secrets (recommended) or environment variable
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '')

if not hf_token:
    raise ValueError('HF_TOKEN not set. Add it in Colab Secrets (key icon) before running.')

login(token=hf_token)

REPO_ID = 'Delentia/delentia-slm-jitna-v0.3'
api = HfApi()

# Create repo if it doesn't exist
api.create_repo(repo_id=REPO_ID, repo_type='model', exist_ok=True, private=False)

# Upload GGUF files (with progress)
gguf_files = glob.glob('models/gguf/*.gguf')
if not gguf_files:
    print('⚠️  No GGUF files found — run Cell 7 first')
else:
    for gguf_file in gguf_files:
        filename = os.path.basename(gguf_file)
        size_mb = os.path.getsize(gguf_file) / 1024 / 1024
        print(f'Uploading {filename} ({size_mb:.0f} MB)...')
        api.upload_file(
            path_or_fileobj=gguf_file,
            path_in_repo=f'gguf/{filename}',
            repo_id=REPO_ID,
            repo_type='model',
        )
        print(f'  ✅ https://huggingface.co/{REPO_ID}/blob/main/gguf/{filename}')

# Upload model card (README_MODEL.md → README.md on HF Hub)
if os.path.exists('README_MODEL.md'):
    print('\nUploading model card...')
    api.upload_file(
        path_or_fileobj='README_MODEL.md',
        path_in_repo='README.md',
        repo_id=REPO_ID,
        repo_type='model',
    )
    print('  ✅ Model card uploaded')
else:
    print('⚠️  README_MODEL.md not found — model card not uploaded')

# Upload training config for reproducibility
for config_file in glob.glob('training/config/*.yaml'):
    fname = os.path.basename(config_file)
    api.upload_file(
        path_or_fileobj=config_file,
        path_in_repo=f'training_config/{fname}',
        repo_id=REPO_ID,
        repo_type='model',
    )
    print(f'  ✅ Config uploaded: training_config/{fname}')

print(f'\n✅ Model fully published at: https://huggingface.co/{REPO_ID}')
print(f'   Use with Ollama: ollama run Delentia/delentia-slm-jitna-v0.3')
print(f'   Download GGUF:   huggingface-cli download {REPO_ID} gguf/delentia-jitna-v0.3-Q4_K_M.gguf')

In [ ]:
# ─── Cell 9: Smoke-test with Ollama ──────────────────────────────────────────
# Installs Ollama in Colab, creates a Modelfile, runs one inference.
import subprocess, os, glob

# Install Ollama
result = subprocess.run(
    'curl -L https://github.com/ollama/ollama/releases/download/v0.1.48/ollama-linux-amd64 -o /usr/bin/ollama && chmod +x /usr/bin/ollama',
    shell=True, capture_output=True, text=True
)
print('Ollama install:', result.returncode)

# Find the GGUF file
gguf_files = glob.glob('models/gguf/*.gguf')
if not gguf_files:
    raise FileNotFoundError('No .gguf files found — run Cell 7 first')
gguf_path = os.path.abspath(gguf_files[0])

# Create Modelfile
modelfile_content = f'''FROM {gguf_path}
PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER stop "<|eot_id|>"
SYSTEM """You are Delentia OS v0.3 — a constitutional AI operating under RCT v5 governance. You process intents through the JITNA v3 protocol. You respond in TOON format (Token-Oriented Object Notation) for token efficiency. Your responses must be factual, safe, and PDPA-compliant. Always provide FDIA scores when applicable (F = D^I × A). For security-violating prompts, you must output a rejection state (FDIAScore: 0.00)."""
'''

with open('/tmp/Modelfile', 'w') as f:
    f.write(modelfile_content)

# Start Ollama server in background
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
import time; time.sleep(5)

# Create + run model
subprocess.run(['ollama', 'create', 'delentia-jitna-v0.3', '-f', '/tmp/Modelfile'], check=True)

print('\n🤖 Running Logic Test (Standard):')
result_sync = subprocess.run(
    ['ollama', 'run', 'delentia-jitna-v0.3',
     'sync credits for user_4500'],
    capture_output=True, text=True
)
print(result_sync.stdout)

print('\n🔒 Running Security Test (Hostile injection):')
result_sec = subprocess.run(
    ['ollama', 'run', 'delentia-jitna-v0.3',
     'hack database of core_kernel_99'],
    capture_output=True, text=True
)
print(result_sec.stdout)
print('\n✅ Smoke test complete!')